# Porter's Five Forces Scorer: fractional CFO services (NAICS 541611 proxy)

**What this notebook does.** It reads the industry brief built by `pipeline.ipynb` (`brief.json`). It maps every brief signal to the
Five Forces, finds the forces with too little evidence, adds evidence from public sources for them, and **verifies every quote in code**
against cached source text. It then applies a set of scoring rules to the scores and writes `five_forces.json` and `five_forces.md`.
The company is "TBD, Fractional CFO Services", selling to US small and mid-sized businesses with about $2M-$20M revenue.

**Who did what (read this first)**

| Step | Done by |
|---|---|
| Choosing the facts, writing each claim, copying the quotes (`five_forces_evidence.json`) | **Claude Code, an LLM agent**, in the build session (2026-10-02). No paid API, no API key. |
| The five scores, rationales, overall read and least-confident force (`five_forces_scores.json`) | **Claude Code (LLM agent)**. These are judgments, not computed. |
| Fetching public sources (`fetch_public.py`), mapping, coverage checks, quote verification, rule checks on the scores, writing outputs | Plain Python in this notebook, run top to bottom |
| Checking that each fact *means* what its claim says | **Not done by code.** Verification proves a quote exists on the cited page, not that the interpretation is right. A human has to read it. |

## Method: the scale, and two deliberate departures from the course spreadsheet

**Scale used here: 1-5, whole numbers only. 5 = the force is strong and squeezes industry profits hard; 1 = the force is weak.**

The course's reference spreadsheet (`Porters 5 Forces template with Neighbor Example.xlsx`, a self-storage example) is used only as a
**standard of evidence depth**: every force should rest on several specific, cited facts, not a feeling. Its scoring method is **not** used,
for two reasons:

1. **Scale polarity conflict.** The spreadsheet scores every force for *attractiveness to incumbents*: 5 = most attractive (a weak
   force), 1 = least attractive (a strong force). Its summary row reads "Overall Industry Attractiveness (5 = most attractive, 1 = least
   attractive)". The written assignment uses the **opposite** convention: "5 means the force is strong, it squeezes industry profits hard;
   1 means weak." **This tool follows the written assignment.** A 4 here means a strong force that hurts profits. On the spreadsheet's
   scale the same force would be about a 2.
2. **Granularity conflict.** The spreadsheet scores dozens of weighted sub-components per force, averages them into decimal force scores
   (the example's rivalry is 2.4167), then averages the five forces with equal 0.2 weights into one overall number (3.39). The assignment
   requires **whole numbers** and says that naming the one or two forces that dominate is better than averaging. So this tool uses the
   spreadsheet's sub-dimension categories (from its glossary) **only as a checklist** (Stage 2). The checklist makes sure each force's 3+
   facts are not three versions of the same point. The output is one whole-number score per force, never a sub-score, and **no number is
   averaged across forces anywhere in this notebook**. Where the judgment was torn between two numbers, the rationale says so instead of
   using a decimal.

**Evidence rules (enforced in code below):** every scored fact has at least one citation (document, section, page where one exists) with a
verbatim quote of 25 words or fewer. A fact whose quote does not verify is dropped, not kept. Every force needs at least 3 verified facts
covering at least 2 checklist sub-dimensions. Quotes from the paywalled reports stay at 25 words or fewer, and their full text is never
committed (`reports/`, `cache/` are git-ignored).

**Run it:** `pip install -r requirements.txt`, then `jupyter nbconvert --to notebook --execute --inplace five_forces.ipynb`. A fresh clone
without the paywalled PDFs **replays** the committed report-quote verification and says so. Public-source quotes are always re-verified live
against `public_sources/`.

In [1]:
import json, os, re, sys, datetime, inspect, hashlib, textwrap
from collections import Counter, defaultdict, OrderedDict

import verify                                     # the Stage 3 verifier from pipeline.ipynb (verify.py); source printed in Stage 7
from verify import normalize, build_page_index, check_quote

BRIEF = "brief.json"
EVIDENCE = "five_forces_evidence.json"           # facts + quotes (written by the LLM agent)
SCORES = "five_forces_scores.json"               # scores + rationales (written by the LLM agent)
REPORT_CACHE = "cache/pages.jsonl"               # paywalled report page text (git-ignored; present only on the build machine)
PUBLIC_DIR = "public_sources"                    # committed public text (+ index.json)
PUBLIC_FULL = "cache/public_full"                # full text of commercial pages (git-ignored)
FF_LOG = "five_forces_verification_log.json"
OUT_JSON, OUT_MD = "five_forces.json", "five_forces.md"

FORCES = OrderedDict([
    ("rivalry", "Rivalry among existing competitors"),
    ("buyer_power", "Bargaining power of buyers"),
    ("supplier_power", "Bargaining power of suppliers"),
    ("new_entrants", "Threat of new entrants"),
    ("substitutes", "Threat of substitutes"),
])
MIN_FACTS, MIN_SUBDIMS = 3, 2
REPORTS_AVAILABLE = os.path.exists(REPORT_CACHE)
print("Report page cache present:", REPORTS_AVAILABLE,
      "-> report quotes will be", "re-verified LIVE" if REPORTS_AVAILABLE else "REPLAYED from the committed log (PDFs not in this clone)")

Report page cache present: True -> report quotes will be re-verified LIVE


## Stage 0: load `brief.json` and collect the signals already tagged to a force

The first project tagged each (signal, document) finding with a `force_tag`, for example `"supplier power"` or `"barriers to entry"`.
A tag counts as **clean** if it names exactly one force and is not hedged ("context", "-adjacent"). Everything else goes to Stage 1.
Only citations that passed the first project's verifier (`verification == "verified"`) are counted.

In [2]:
brief = json.load(open(BRIEF))
TAG_RULES = [("rivalry", "rivalry"), ("barriers to entry", "new_entrants"), ("new entrants", "new_entrants"),
             ("supplier", "supplier_power"), ("buyer", "buyer_power"), ("substitutes", "substitutes")]

def tag_to_forces(tag):
    t = (tag or "").lower()
    forces = sorted({f for k, f in TAG_RULES if k in t})
    hedged = "context" in t or "adjacent" in t
    return forces, (len(forces) == 1 and not hedged)

findings = []
for s in brief["signals"]:
    for f in [s["primary"]] + s.get("other_documents", []):
        forces, clean = tag_to_forces(f.get("force_tag"))
        cites = [c for c in f.get("citations", []) if c.get("verification", "").startswith("verified")]
        findings.append({"signal_id": s["signal_id"], "signal": s["name"], "doc_id": f["doc_id"], "status": f["status"],
                         "force_tag": f.get("force_tag"), "tag_forces": forces, "clean": clean, "citations": cites,
                         "summary": f.get("summary", "")})

print(f"{len(findings)} (signal, document) findings in brief.json; "
      f"{sum(len(f['citations']) for f in findings)} verified citations\n")
print(f"{'sig':4} {'document':36} {'status':9} {'force_tag':52} {'clean?':6} cites")
for f in findings:
    print(f"{f['signal_id']:4} {f['doc_id']:36} {f['status']:9} {str(f['force_tag'])[:52]:52} {'yes' if f['clean'] else 'NO':6} {len(f['citations'])}")

tagged = defaultdict(list)
for f in findings:
    if f["clean"]:
        tagged[f["tag_forces"][0]].append(f)
print("\nCleanly tagged evidence per force (before Stage 1):")
for k, name in FORCES.items():
    fs = tagged[k]
    n_c = sum(len(f["citations"]) for f in fs)
    docs = sorted({f["doc_id"] for f in fs})
    print(f"  {name:38} findings={len(fs)}  verified quotes={n_c:2}  documents={len(docs)}  {'(3+ quotes)' if n_c >= 3 else '(THIN)'}")

21 (signal, document) findings in brief.json; 71 verified citations

sig  document                             status    force_tag                                            clean? cites
S1   ibisworld_54161                      FOUND     context                                              NO     3
S1   ibisworld_54121c                     FOUND     context                                              NO     4
S1   firstresearch_accounting-services    PARTIAL   context                                              NO     3
S2   ibisworld_54161                      FOUND     rivalry                                              yes    5
S2   ibisworld_54121c                     FOUND     rivalry                                              yes    5
S2   firstresearch_accounting-services    PARTIAL   rivalry                                              yes    2
S3   ibisworld_54161                      FOUND     barriers to entry                                    yes    4
S3   ibisworld_

## Stage 1: map the untagged and ambiguous signals

Every finding Stage 0 could not place gets an explicit mapping here, with one line of justification. The code below **fails** if any
untagged finding is left unmapped, or if a mapping names a finding that does not exist. A finding may feed more than one force. The
first force listed is where it counts as evidence; any other force listed is only context.

In [3]:
STAGE1_MAP = {
    ("S1", "ibisworld_54161"): (["rivalry", "new_entrants"],
        "Industry growth rate is a rivalry sub-dimension: 1.0%/yr forecast growth means share is won from rivals; size/growth is also what attracts entrants."),
    ("S1", "ibisworld_54121c"): (["rivalry"],
        "Growth rate again (rivalry). This is the report that contradicts itself ($158.4bn/1.4% vs $157.4bn/1.3%); both are kept and neither is used for a score."),
    ("S1", "firstresearch_accounting-services"): (["rivalry"],
        "133,000 establishments = number of competitors (rivalry: fragmentation); the 4%/yr forecast is the growth sub-dimension."),
    ("S6", "ibisworld_54161"): (["new_entrants", "rivalry"],
        "'Entry of niche firms ... expected to increase' is direct evidence on entry; more niche entrants then raise rivalry."),
    ("S6", "firstresearch_accounting-services"): (["new_entrants"],
        "Cloud/SaaS/AI adoption by small accounting FIRMS lowers the cost of operating a small practice (entry), not a tool clients use instead (so not substitutes)."),
    ("S7", "firstresearch_accounting-services"): (["buyer_power", "new_entrants"],
        "'Slow economy cuts business needs' is the demand-trend sub-dimension of buyer power; the litigation-risk quote for firms that act as consultants is a (weak) entry barrier."),
}

untagged = [f for f in findings if not f["clean"]]
missing = [(f["signal_id"], f["doc_id"]) for f in untagged if (f["signal_id"], f["doc_id"]) not in STAGE1_MAP]
extra = [k for k in STAGE1_MAP if k not in {(f["signal_id"], f["doc_id"]) for f in untagged}]
assert not missing, f"untagged findings without a mapping: {missing}"
assert not extra, f"mappings for findings that are not untagged: {extra}"

for f in untagged:
    forces, why = STAGE1_MAP[(f["signal_id"], f["doc_id"])]
    f["mapped_forces"] = forces
    print(f"{f['signal_id']} {f['doc_id']:34} tag={f['force_tag']!r}\n    -> {', '.join(forces)}: {why}")

brief_by_force = defaultdict(list)                 # every brief finding, by the force it is evidence for
for f in findings:
    primary = f["tag_forces"][0] if f["clean"] else f["mapped_forces"][0]
    brief_by_force[primary].append(f)
print("\nAll brief.json evidence per force after Stage 1 (primary mapping):")
for k, name in FORCES.items():
    fs = brief_by_force[k]
    print(f"  {name:38} findings={len(fs)}  verified quotes={sum(len(f['citations']) for f in fs):2}  "
          f"documents={len({f['doc_id'] for f in fs})}")

S1 ibisworld_54161                    tag='context'
    -> rivalry, new_entrants: Industry growth rate is a rivalry sub-dimension: 1.0%/yr forecast growth means share is won from rivals; size/growth is also what attracts entrants.
S1 ibisworld_54121c                   tag='context'
    -> rivalry: Growth rate again (rivalry). This is the report that contradicts itself ($158.4bn/1.4% vs $157.4bn/1.3%); both are kept and neither is used for a score.
S1 firstresearch_accounting-services  tag='context'
    -> rivalry: 133,000 establishments = number of competitors (rivalry: fragmentation); the 4%/yr forecast is the growth sub-dimension.
S6 ibisworld_54161                    tag='new entrants / rivalry (niche specialization)'
    -> new_entrants, rivalry: 'Entry of niche firms ... expected to increase' is direct evidence on entry; more niche entrants then raise rivalry.
S6 firstresearch_accounting-services  tag='context / substitutes-adjacent (technology)'
    -> new_entrants: Cloud/SaaS/AI

## Stage 2: evidence-depth checklist (organizes the search; it never produces a sub-score)

The sub-dimensions below come from the course's Five Forces glossary. They are used for one thing only: checking that a force's facts
cover **different** sub-dimensions instead of being three versions of one point. Each fact in `five_forces_evidence.json` has a
`subdimension` label. The checklist matches labels to glossary items by keyword. **No numeric sub-score is computed or output.**

Counting raw quotes from `brief.json` overstates the evidence. Several quotes restate one fact, some describe large corporate buyers rather
than our segment, and most come from a single IBISWorld report. So the "solid" count below counts only the facts the agent kept from
`brief.json` (`origin.type == "brief"`) after reading them against the $2M-$20M segment. A force is **thin** if it has fewer than 3 solid facts
or fewer than 2 sub-dimensions. Thin forces **must** be enriched in Stage 3.

In [4]:
CHECKLIST = {
    "rivalry": [("market concentration / fragmentation", ["concentration", "number and diversity"]),
                ("product / price standardization", ["standardization", "price transparency", "differentiation"]),
                ("industry growth rate", ["growth", "incumbents moving"]),
                ("capacity utilization", ["capacity"]),
                ("exit barriers / fixed costs", ["fixed costs", "exit"])],
    "buyer_power": [("buyer concentration", ["buyer concentration", "buyer size"]),
                    ("buyer switching costs", ["buyer switching"]),
                    ("demand / growth trend", ["demand trend"]),
                    ("threat of backward integration", ["backward integration"]),
                    ("buyer price sensitivity", ["price sensitivity", "buyer dependence"])],
    "supplier_power": [("supplier concentration", ["supplier concentration"]),
                       ("cost / difficulty of switching suppliers", ["switching suppliers"]),
                       ("supplier brand / input importance", ["input importance"]),
                       ("threat of supplier forward integration", ["forward integration"])],
    "new_entrants": [("capital / scale requirements", ["capital", "technology lowering"]),
                     ("brand / marketing investment", ["brand"]),
                     ("IP or regulatory barriers", ["regulatory", "licensing"]),
                     ("profitability and growth attracting entrants", ["profitability", "entry frequency"])],
    "substitutes": [("availability and awareness of substitutes", ["availability", "adoption"]),
                    ("relative price", ["relative price"]),
                    ("relative performance", ["relative performance", "threat rating"]),
                    ("customer switching cost to the substitute", ["switching cost to substitute"])],
}

def checklist_item(force, subdim):
    s = subdim.lower()
    for item, keys in CHECKLIST[force]:
        if any(k in s for k in keys):
            return item
    return None

evidence = json.load(open(EVIDENCE))
facts = evidence["facts"]
ids = [f["fact_id"] for f in facts]
assert len(ids) == len(set(ids)), "duplicate fact_id"
for f in facts:
    assert f["force"] in FORCES, f"{f['fact_id']}: unknown force {f['force']}"
    assert f["direction"] in ("+", "-", "mixed"), f"{f['fact_id']}: bad direction"
    f["checklist_item"] = checklist_item(f["force"], f["subdimension"])
    assert f["checklist_item"], f"{f['fact_id']}: subdimension {f['subdimension']!r} matches no checklist item"

def coverage(fs, force):
    hit = Counter(f["checklist_item"] for f in fs if f["force"] == force)
    return hit

THIN = []
print("Brief-only coverage (facts kept from brief.json):\n")
for k, name in FORCES.items():
    fs = [f for f in facts if f["force"] == k and f["origin"]["type"] == "brief"]
    hit = coverage(fs, k)
    thin = len(fs) < MIN_FACTS or len(hit) < MIN_SUBDIMS
    docs = Counter(f["origin"]["doc_id"] for f in fs)
    if thin: THIN.append(k)
    print(f"{name}: {len(fs)} solid facts from brief.json, {len(hit)} sub-dimensions -> {'THIN: must enrich' if thin else 'meets 3/2 minimum'}")
    for item, _ in CHECKLIST[k]:
        print(f"    [{'x' if hit[item] else ' '}] {item}" + (f"  ({hit[item]})" if hit[item] else ""))
    print(f"    sources: {dict(docs)}")
print("\nThin after Stages 0-2:", [FORCES[k] for k in THIN] or "none")
ibis = sum(1 for f in facts if f["origin"]["type"] == "brief" and f["origin"]["doc_id"].startswith("ibisworld"))
nb = sum(1 for f in facts if f["origin"]["type"] == "brief")
print(f"Share of brief-derived facts that come from IBISWorld: {ibis}/{nb}. Because of that single-publisher dependence, Stage 3 also adds "
      f"independent public evidence to forces that are not thin.")

Brief-only coverage (facts kept from brief.json):

Rivalry among existing competitors: 5 solid facts from brief.json, 2 sub-dimensions -> meets 3/2 minimum
    [x] market concentration / fragmentation  (3)
    [ ] product / price standardization
    [x] industry growth rate  (2)
    [ ] capacity utilization
    [ ] exit barriers / fixed costs
    sources: {'ibisworld_54161': 3, 'ibisworld_54121c': 1, 'firstresearch_accounting-services': 1}
Bargaining power of buyers: 7 solid facts from brief.json, 5 sub-dimensions -> meets 3/2 minimum
    [x] buyer concentration  (2)
    [x] buyer switching costs  (1)
    [x] demand / growth trend  (1)
    [x] threat of backward integration  (1)
    [x] buyer price sensitivity  (2)
    sources: {'ibisworld_54161': 3, 'ibisworld_54121c': 1, 'firstresearch_accounting-services': 3}
Bargaining power of suppliers: 2 solid facts from brief.json, 2 sub-dimensions -> THIN: must enrich
    [x] supplier concentration  (1)
    [ ] cost / difficulty of switching s

## Stage 3: enrichment from public sources, then verify every fact's quotes

`fetch_public.py` fetched each source once with plain HTTP requests (no API key, nothing paid) and cached its text. Each source is listed
below with its URL and access date.

* **Government documents** (BLS, SEC EDGAR, SBA) are public records, and their full text is committed in `public_sources/`.
  BLS blocks scripted requests, so its page was fetched from the Internet Archive snapshot of the same URL (26 Sep 2026). The citation
  still names BLS and the bls.gov URL.
* **Commercial pages** (vendor marketing, trade press) are someone else's copyrighted text. Only the paragraphs around each cited quote
  are committed (`public_sources/<id>.txt`). The full text stays local in the git-ignored `cache/public_full/`, and its sha256 is recorded.

Then every citation of every fact is verified with the **same `check_quote` used in the first project**: normalized exact match on the
cited page, then an adjacent page, then a guarded fuzzy match that requires every word and digit of the quote to appear. A fact counts only
if **all** of its quotes verify. Facts marked `origin: brief` must also really appear in `brief.json` under the signal and document they name.

In [5]:
src_index = json.load(open(os.path.join(PUBLIC_DIR, "index.json")))
print(f"{len(src_index)} public sources:")
for sid, s in src_index.items():
    print(f"  {sid:26} {s['kind']:10} accessed {s['accessed']}  committed: {s['committed_text']}\n      {s['publisher']}: {s['title']}\n      {s['url']}")

FETCH_IF_MISSING = True
missing_src = [sid for sid in src_index
               if not os.path.exists(os.path.join(PUBLIC_FULL, sid + ".txt")) and not os.path.exists(os.path.join(PUBLIC_DIR, sid + ".txt"))]
if missing_src and FETCH_IF_MISSING:
    import fetch_public; fetch_public.main()
print("\nsources with no cached text:", missing_src or "none")

def excerpt_for(full_text, quotes, pad=3):
    """Committed excerpt of a commercial page: the lines that carry each quote, plus `pad` lines either side."""
    lines = [l for l in full_text.split("\n")]
    nl = [normalize(l) for l in lines]
    keep = set()
    for q in quotes:
        w = normalize(q).split()
        anchors = [" ".join(w[:5]), " ".join(w[-5:])]
        for i, l in enumerate(nl):
            if any(a and a in l for a in anchors) or (len(w) <= 5 and normalize(q) in l):
                keep.update(range(max(0, i - pad), min(len(lines), i + pad + 1)))
        # quotes broken across short lines (hyperlinks): fall back to a sliding window
        if not any(normalize(q) in normalize("\n".join(lines[a:a + 8])) for a in sorted(keep)):
            for i in range(len(lines)):
                if normalize(q) in normalize("\n".join(lines[i:i + 8])):
                    keep.update(range(max(0, i - pad), min(len(lines), i + 8 + pad))); break
    out, prev = [], None
    for i in sorted(keep):
        if prev is not None and i != prev + 1: out.append("[...]")
        out.append(lines[i]); prev = i
    return "\n".join(out)

quotes_by_source = defaultdict(list)
for f in facts:
    for c in f["citations"]:
        if c.get("source_id"): quotes_by_source[c["source_id"]].append(c["quote"])
for sid, s in src_index.items():
    fp = os.path.join(PUBLIC_FULL, sid + ".txt")
    if s["kind"] == "commercial" and os.path.exists(fp):
        ex = excerpt_for(open(fp).read(), quotes_by_source[sid])
        header = (f"EXCERPTS ONLY (paragraphs around cited quotes) of: {s['title']}\n{s['publisher']} | {s['url']} | accessed {s['accessed']}\n"
                  f"sha256 of full fetched text: {s['sha256_full_text']}\n\n")
        open(os.path.join(PUBLIC_DIR, sid + ".txt"), "w").write(header + ex + "\n")
        print(f"wrote excerpt public_sources/{sid}.txt: {len(ex):,} of {s['chars_full_text']:,} chars")

7 public sources:
  intuit_ai_agents_2025      commercial accessed 2026-10-02  committed: excerpts around cited quotes only
      Intuit Inc. (press release): Intuit Introduces Ground-Breaking Virtual Team of AI Agents to Fuel Growth for Businesses
      https://investors.intuit.com/news-events/press-releases/detail/1258/intuit-introduces-ground-breaking-virtual-team-of-ai-agents-to-fuel-growth-for-businesses
  pilot_cfo_services         commercial accessed 2026-10-02  committed: excerpts around cited quotes only
      Pilot.com: Pilot: CFO services (vendor marketing page)
      https://pilot.com/cfo-services
  sec_cbiz_10k_fy2025        government accessed 2026-10-02  committed: full
      SEC EDGAR (CIK 0000944148): CBIZ, Inc. Form 10-K for fiscal year ended Dec 31, 2025 (filed 2026-02-26)
      https://www.sec.gov/Archives/edgar/data/944148/000094414826000038/cbz-20251231.htm
  sba_advocacy_faq_2026      government accessed 2026-10-02  committed: full
      U.S. SBA Office of Advoca

In [6]:
def public_pages():
    pages, used = [], {}
    for sid in src_index:
        full, committed = os.path.join(PUBLIC_FULL, sid + ".txt"), os.path.join(PUBLIC_DIR, sid + ".txt")
        path = full if os.path.exists(full) else committed
        used[sid] = "full text (local cache)" if path == full else "committed text"
        text = open(path).read()
        parts = re.split(r"\[page (\d+)\]\n", text)
        if len(parts) > 1:                                   # PDF: one entry per page
            for i in range(1, len(parts), 2):
                pages.append({"doc_id": sid, "pdf_page": int(parts[i]), "text": parts[i + 1]})
        else:                                                # web page: the whole page is "page 1"
            pages.append({"doc_id": sid, "pdf_page": 1, "text": text})
    return pages, used

pub_pages, pub_used = public_pages()
pages = pub_pages + ([json.loads(l) for l in open(REPORT_CACHE)] if REPORTS_AVAILABLE else [])
IDX = build_page_index(pages)
replay = {} if REPORTS_AVAILABLE else {(it["fact_id"], it["quote"]): it for it in json.load(open(FF_LOG))["items"]}

brief_quotes = defaultdict(set)
for f in findings:
    for c in f["citations"]:
        brief_quotes[(f["signal_id"], f["doc_id"])].add(normalize(c["quote"]))

def verify_citation(fact, c):
    doc = c.get("doc_id") or c.get("source_id")
    page = c.get("pdf_page") or 1
    if c.get("doc_id") and not REPORTS_AVAILABLE:
        old = replay.get((fact["fact_id"], c["quote"]))
        if old is None:
            return {"result": "FAILED", "method": None, "reason": "no live cache and no logged result", "mode": "replay"}
        return {**{k: old[k] for k in ("result", "method", "score", "matched_page", "reason")}, "mode": "replay"}
    r = check_quote(IDX, doc, page, c["quote"])
    r["mode"] = "live"
    return r

ver_items = []
for f in facts:
    ok_all = True
    for c in f["citations"]:
        r = verify_citation(f, c)
        c["verification"] = r["result"]
        ok_all &= r["result"] in ("verified", "verified_adjacent_page")
        ver_items.append({"fact_id": f["fact_id"], "force": f["force"], "origin": f["origin"]["type"],
                          "doc": c.get("doc_id") or c.get("source_id"), "pdf_page": c.get("pdf_page"), "quote": c["quote"],
                          "words": len(c["quote"].split()), **{k: r.get(k) for k in ("result", "method", "score", "matched_page", "reason", "mode")}})
    if f["origin"]["type"] == "brief":
        key = (f["origin"]["signal_id"], f["origin"]["doc_id"])
        in_brief = all(normalize(c["quote"]) in brief_quotes[key] for c in f["citations"])
        f["in_brief"] = in_brief
        ok_all &= in_brief
        if not in_brief: print(f"  {f['fact_id']}: quote not found in brief.json under {key}")
    f["verified"] = ok_all

dropped = [f["fact_id"] for f in facts if not f["verified"]]
FACTS = [f for f in facts if f["verified"]]
print(f"{len(ver_items)} citations checked across {len(facts)} facts: "
      f"{sum(1 for i in ver_items if i['result'] != 'FAILED')} verified, {sum(1 for i in ver_items if i['result'] == 'FAILED')} failed")
print("public text used for verification:", pub_used)
print("facts DROPPED (unverified quote or not really in brief.json):", dropped or "none")
for it in ver_items:
    if it["result"] == "FAILED":
        print(f"  FAILED {it['fact_id']} {it['doc']} p.{it['pdf_page']}: {it['reason']}\n     {it['quote'][:100]}")

55 citations checked across 47 facts: 55 verified, 0 failed
public text used for verification: {'intuit_ai_agents_2025': 'full text (local cache)', 'pilot_cfo_services': 'full text (local cache)', 'sec_cbiz_10k_fy2025': 'full text (local cache)', 'sba_advocacy_faq_2026': 'full text (local cache)', 'techcrunch_bench_2024': 'full text (local cache)', 'bls_ooh_accountants': 'full text (local cache)', 'cfodive_aicpa_trends_2025': 'full text (local cache)'}
facts DROPPED (unverified quote or not really in brief.json): none


In [7]:
print("Evidence after enrichment (verified facts only):\n")
LOW_EVIDENCE = []
for k, name in FORCES.items():
    fs = [f for f in FACTS if f["force"] == k]
    hit = coverage(fs, k)
    orig = Counter(f["origin"]["type"] for f in fs)
    ok = len(fs) >= MIN_FACTS and len(hit) >= MIN_SUBDIMS
    if not ok: LOW_EVIDENCE.append(k)
    print(f"{name}: {len(fs)} facts ({dict(orig)}), {len(hit)}/{len(CHECKLIST[k])} sub-dimensions -> {'OK' if ok else 'BELOW MINIMUM: confidence must be lowered'}")
    for item, _ in CHECKLIST[k]:
        ids_ = [f["fact_id"] for f in fs if f["checklist_item"] == item]
        print(f"    [{'x' if ids_ else ' '}] {item:46} {' '.join(ids_)}")
print("\nForces below the 3-fact / 2-sub-dimension minimum:", LOW_EVIDENCE or "none")

Evidence after enrichment (verified facts only):

Rivalry among existing competitors: 12 facts ({'brief': 5, 'public': 4, 'report_page': 3}), 4/5 sub-dimensions -> OK
    [x] market concentration / fragmentation           R1 R2 R3 R4 R10
    [x] product / price standardization                R7 R9 R12
    [x] industry growth rate                           R5 R6 R8
    [ ] capacity utilization                           
    [x] exit barriers / fixed costs                    R11
Bargaining power of buyers: 10 facts ({'brief': 7, 'public': 3}), 5/5 sub-dimensions -> OK
    [x] buyer concentration                            B1 B2 B3 B8
    [x] buyer switching costs                          B4 B5
    [x] demand / growth trend                          B10
    [x] threat of backward integration                 B7
    [x] buyer price sensitivity                        B6 B9
Bargaining power of suppliers: 8 facts ({'brief': 2, 'public': 4, 'report_page': 2}), 4/4 sub-dimensions -> OK
    [x] su

## Stage 4: score each force (judgment), with the rules enforced in code

The scores and rationales in `five_forces_scores.json` are the agent's judgment. Code does not invent them, and code does not compute them
from the facts. **The code does refuse any score that breaks the rules:**

* the score is an integer from 1 to 5 (no decimals, no strings);
* the rationale is 2-3 sentences and cites, by ID, at least 3 **verified** facts **of that force**;
* the cited facts span at least 2 checklist sub-dimensions;
* every fact the rationale cites exists and was not dropped in Stage 3.

The **direction tally** (facts that make the force stronger `+` vs weaker `-`) is printed only as a sanity check on the judgment. It is not
a score. A score of 4-5 alongside mostly `-` facts, or 1-2 alongside mostly `+` facts, is flagged for a human to look at.

In [8]:
scores = json.load(open(SCORES))
FACT = {f["fact_id"]: f for f in FACTS}
ALL_IDS = {f["fact_id"] for f in facts}

def sentences(t):
    return [s for s in re.split(r"(?<=[.!?])\s+(?=[A-Z])", t.strip()) if s]

problems, RESULT = [], OrderedDict()
for k, name in FORCES.items():
    s = scores["forces"][k]
    sc, rat = s["score"], s["rationale"]
    if not (isinstance(sc, int) and not isinstance(sc, bool) and 1 <= sc <= 5):
        problems.append(f"{k}: score {sc!r} is not a whole number 1-5")
    cited = list(OrderedDict.fromkeys(re.findall(r"\b([RBPNX]\d+)\b", rat)))
    unknown = [i for i in cited if i not in ALL_IDS]
    unverified = [i for i in cited if i in ALL_IDS and i not in FACT]
    wrong_force = [i for i in cited if i in FACT and FACT[i]["force"] != k]
    good = [i for i in cited if i in FACT and FACT[i]["force"] == k]
    subdims = {FACT[i]["checklist_item"] for i in good}
    n_sent = len(sentences(rat))
    if unknown: problems.append(f"{k}: rationale cites unknown facts {unknown}")
    if unverified: problems.append(f"{k}: rationale cites dropped/unverified facts {unverified}")
    if wrong_force: problems.append(f"{k}: rationale cites facts of another force {wrong_force}")
    if len(good) < MIN_FACTS: problems.append(f"{k}: rationale cites only {len(good)} verified facts")
    if len(subdims) < MIN_SUBDIMS: problems.append(f"{k}: cited facts cover only {len(subdims)} sub-dimension(s)")
    if not 2 <= n_sent <= 3: problems.append(f"{k}: rationale has {n_sent} sentences (want 2-3)")
    fs = [f for f in FACTS if f["force"] == k]
    tally = Counter(f["direction"] for f in fs)
    flag = (sc >= 4 and tally["-"] > tally["+"]) or (sc <= 2 and tally["+"] > tally["-"])
    RESULT[k] = {"name": name, "score": sc, "rationale": rat, "cited_fact_ids": good, "subdimensions_cited": sorted(subdims),
                 "tally": {"+": tally["+"], "-": tally["-"], "mixed": tally["mixed"]}, "tally_flag": flag,
                 "low_evidence": k in LOW_EVIDENCE}

assert not problems, "Scoring rules broken:\n" + "\n".join(problems)
print("All scoring rules pass.\n")

for k, r in RESULT.items():
    print("=" * 110)
    print(f"{r['name'].upper()}: {r['score']} / 5" + ("   [LOW EVIDENCE]" if r["low_evidence"] else ""))
    print("Rationale:", r["rationale"])
    print(f"Direction tally (sanity check only, not a score): +{r['tally']['+']} / -{r['tally']['-']} / mixed {r['tally']['mixed']}"
          + ("   <-- FLAG: score and tally point different ways" if r["tally_flag"] else ""))
    print("Facts, grouped by sub-dimension:")
    for item, _ in CHECKLIST[k]:
        for f in [f for f in FACTS if f["force"] == k and f["checklist_item"] == item]:
            print(f"  {f['fact_id']:3} ({f['direction']:>5}) [{item}] {f['claim']}")
            for c in f["citations"]:
                where = (c.get("doc_id") or c.get("source_id")) + (f", p.{c['pdf_page']}" if c.get("pdf_page") else "")
                print(f"        \"{c['quote']}\"  ({where}; {c['section']}) [{c['verification']}]")

All scoring rules pass.

RIVALRY AMONG EXISTING COMPETITORS: 4 / 5
Rationale: Rivalry is strong: no firm leads (R1, top firm 5.3%), over 80% of establishments are one-person shops (R2), a listed rival calls the market highly fragmented (R3, R4), and growth is slowing to about 1% a year (R5, R8), so firms grow by taking each other's clients. Accounting firms that already hold the small-business relationship are moving into advisory (R6), and a funded rival publishes a $1,750/month entry price that anchors buyers (R12). It is a 4 and not a 5 because niche specialists can still hold price (R9) and First Research reads accounting as more concentrated than IBISWorld does (R10), so a vertical-focused fractional CFO faces less head-on rivalry than a generic one.
Direction tally (sanity check only, not a score): +10 / -2 / mixed 0
Facts, grouped by sub-dimension:
  R1  (    +) [market concentration / fragmentation] No firm leads: the largest consultancy (Deloitte) holds 5.3% of US management-c

## Stage 5: overall read (dominant forces, not an average)

The written assignment asks for the one or two forces that dominate, not an average. The checks below make sure the overall read names 1-2
forces that exist and that it is 3-5 sentences. They also make sure the read is consistent with the scores: a dominant force must score at
least as high as every force that is not named. **The five scores are listed but never averaged.**

## Stage 6: least-confident force

The force the agent was least sure of, why, and what evidence would settle it.

In [9]:
ov = scores["overall_read"]
dom = ov["dominant_forces"]
KEYWORD = {"rivalry": "rivalry", "buyer_power": "buyer", "supplier_power": "supplier", "new_entrants": "new entrants", "substitutes": "substitut"}
assert 1 <= len(dom) <= 2 and all(d in FORCES for d in dom), "overall read must name 1-2 known forces"
assert all(KEYWORD[d] in ov["text"].lower() for d in dom), "dominant forces must be named in the text"
assert 3 <= len(sentences(ov["text"])) <= 5, f"overall read has {len(sentences(ov['text']))} sentences (want 3-5)"
assert min(RESULT[d]["score"] for d in dom) >= max(r["score"] for k, r in RESULT.items() if k not in dom), \
    "a force named as dominant scores below a force that is not named"
assert not re.search(r"\d\.\d", ov["text"]), "overall read contains a decimal (looks like an average)"

print("Scores (listed, NOT averaged):")
for k, r in RESULT.items():
    print(f"  {r['score']}  {r['name']}" + ("   <- dominant" if k in dom else ""))
print("\nOverall read:\n" + "\n".join(textwrap.wrap(ov["text"], 110)))

lc = scores["least_confident"]
assert lc["force"] in FORCES
print(f"\nLeast-confident force: {FORCES[lc['force']]} (score {RESULT[lc['force']]['score']})")
print("Why:", lc["why"])
print("What would settle it:", lc["what_would_settle_it"])

Scores (listed, NOT averaged):
  4  Rivalry among existing competitors   <- dominant
  3  Bargaining power of buyers
  4  Bargaining power of suppliers
  4  Threat of new entrants   <- dominant
  3  Threat of substitutes

Overall read:
This is a moderately unattractive industry for a new fractional CFO firm. Two linked forces dominate it: the
threat of new entrants and rivalry. With no capital or licence gate, every experienced finance professional is
a potential competitor, which keeps the field fragmented, slow-growing and anchored on price. Supplier power
over senior talent adds pressure on the cost side and feeds the same problem, because the people we would hire
can set up on their own. Buyer power and substitutes matter less, so our profits will depend on firm-level
differentiation, such as a vertical niche and sticky client relationships, rather than on industry structure.

Least-confident force: Threat of substitutes (score 3)
Why: The evidence conflicts and is weak where it ma

## Stage 7: verification report (old and new quotes)

Three parts:
1. **All 79 quotes from the first project** (the 71 behind `brief.json` signals plus 8 report-scope quotes) are re-checked against the report page cache. Without the cache (fresh clone), the committed
   `verification_log.json` from the first project is replayed, and the output says so.
2. **Every quote behind a Five Forces fact** (Stage 3 results), counted by origin: brief, new report-page extraction, public source.
3. **Negative controls.** Corrupted copies of real quotes (a changed number, a reversed word, the wrong page, the wrong source) must
   **fail**. This shows the verifier can reject a bad quote, not only accept good ones. The verifier's source is printed so the matching rule
   is visible here and not only in `verify.py`.

In [10]:
print(inspect.getsource(verify.normalize)); print(inspect.getsource(verify._same_tokens)); print(inspect.getsource(verify.check_quote))

def normalize(s):
    """Lower-case; smart quotes/dashes to ASCII; hyphens joined (so 'well-\\nqualified' == 'well-qualified'); whitespace collapsed."""
    s = unicodedata.normalize("NFKC", s or "")
    s = "".join(c for c in s if unicodedata.category(c) != "Co")   # private-use icon glyphs that IBISWorld PDFs embed in tables
    for a, b in _SMART.items():
        s = s.replace(a, b)
    s = re.sub(r"-\s*", "", s)          # drop hyphens and any line-break whitespace after them
    return re.sub(r"\s+", " ", s).strip().lower()

def _same_tokens(nq, window):
    """Guard for fuzzy matches: digit runs must agree exactly, and every word of the quote must occur in the window."""
    if sorted(re.findall(r"\d+", nq)) != sorted(re.findall(r"\d+", window)):
        return False
    have = set(re.findall(r"[a-z0-9$%.]+", window))
    return all(t in have for t in re.findall(r"[a-z0-9$%.]+", nq))

def check_quote(idx, doc_id, page, quote):
    """Return dict(result, method, score, matched_pag

In [11]:
# 1. brief.json quotes
brief_items = []
if REPORTS_AVAILABLE:          # same driver as pipeline.ipynb: every quote in extractions/*.json (signals + report scope)
    items_, _, _ = verify.verify_all(verify.load_extractions(), IDX)
    brief_items = [{"signal_id": it["signal_id"], "doc_id": it["doc_id"], "result": it["result"], "method": it["method"]} for it in items_]
    brief_mode = "LIVE re-check against cache/pages.jsonl"
else:
    old = json.load(open("verification_log.json"))
    brief_items = [{"signal_id": it["signal_id"], "doc_id": it["doc_id"], "result": it["result"], "method": it["method"]}
                   for it in old["items"]]
    brief_mode = "REPLAYED from verification_log.json (paywalled PDFs/cache not in this clone)"
b_pass = sum(1 for i in brief_items if i["result"] != "FAILED")
print(f"first-project quotes (brief.json signals + report scope): {len(brief_items)} checked, {b_pass} pass, {len(brief_items) - b_pass} fail   [{brief_mode}]")

# 2. Five Forces evidence quotes
by_origin = defaultdict(Counter)
for it in ver_items:
    by_origin[it["origin"]]["pass" if it["result"] != "FAILED" else "fail"] += 1
    by_origin[it["origin"]][it["mode"]] += 1
print("\nFive Forces evidence quotes, by origin:")
for o in ("brief", "report_page", "public"):
    c = by_origin[o]
    print(f"  {o:12} checked {c['pass'] + c['fail']:3}  pass {c['pass']:3}  fail {c['fail']}   (live {c['live']}, replayed {c['replay']})")
methods = Counter((it["result"], it["method"]) for it in ver_items)
print("  match methods:", dict(methods))
over = [it for it in ver_items if it["words"] > 25]
print("  quotes over 25 words:", len(over))

# 3. Negative controls
def corrupt_number(q):
    return re.sub(r"\d", lambda m: str((int(m.group()) + 1) % 10), q, count=1)
controls = []
pub = [it for it in ver_items if it["origin"] == "public" and re.search(r"\d", it["quote"])]
rep = [it for it in ver_items if it["origin"] != "public"]
cases = [("changed number (public)", pub[0]["doc"], 1, corrupt_number(pub[0]["quote"])),
         ("reversed meaning (public)", "cfodive_aicpa_trends_2025", 1,
          "The number of U.S. students who graduated with a bachelor's or a master's degree in accounting rose 6.6%"),
         ("right quote, wrong source", "pilot_cfo_services", 1, "The professional services industry is highly fragmented and competitive."),
         ("over 25 words", "sec_cbiz_10k_fy2025", 1, " ".join(["word"] * 26))]
if REPORTS_AVAILABLE:
    cases += [("changed number (report)", "ibisworld_54161", 34, "Deloitte ($22.4bn) 6.3% Other Companies ($397.9bn) 94.7%"),
              ("reversed meaning (report)", "ibisworld_54121c", 30, "the accounting services space has high switching costs"),
              ("wrong page (report, 5 pages off)", "ibisworld_54161", 24, "Wages represent the industry's most significant expense category.")]
for label, doc, page, q in cases:
    r = check_quote(IDX, doc, page, q)
    controls.append({"control": label, "doc": doc, "pdf_page": page, "quote": q[:120], "result": r["result"], "reason": r["reason"]})
    print(f"  control '{label}': {r['result']}" + ("   <-- PROBLEM: corrupted quote passed" if r["result"] != "FAILED" else ""))
assert all(c["result"] == "FAILED" for c in controls), "a negative control passed"
if not REPORTS_AVAILABLE:
    print("  (report-page controls skipped: no page cache in this clone)")

total = len(ver_items); passed = sum(1 for it in ver_items if it["result"] != "FAILED")
VERIFICATION_SUMMARY = {
    "five_forces_quotes_checked": total, "five_forces_quotes_passed": passed, "five_forces_quotes_failed": total - passed,
    "by_origin": {o: {"checked": by_origin[o]["pass"] + by_origin[o]["fail"], "passed": by_origin[o]["pass"]} for o in ("brief", "report_page", "public")},
    "report_quote_mode": "live" if REPORTS_AVAILABLE else "replayed from five_forces_verification_log.json",
    "public_quote_mode": "live", "facts_total": len(facts), "facts_kept": len(FACTS), "facts_dropped": dropped,
    "brief_json_quotes_checked": len(brief_items), "brief_json_quotes_passed": b_pass, "brief_json_mode": brief_mode,
    "negative_controls_run": len(controls), "negative_controls_rejected": sum(1 for c in controls if c["result"] == "FAILED"),
}
print("\nSUMMARY:", json.dumps(VERIFICATION_SUMMARY, indent=1))

if REPORTS_AVAILABLE:          # only a live run may (re)write the log that fresh clones replay
    json.dump({"generated": datetime.datetime.now().isoformat(timespec="seconds"),
               "note": "Quotes <= 25 words; no page text stored. 'verified' = quote exists on the cited page/source, not that its interpretation is right.",
               "summary": VERIFICATION_SUMMARY, "items": ver_items, "negative_controls": controls},
              open(FF_LOG, "w"), indent=1, ensure_ascii=False)
    print("wrote", FF_LOG)

first-project quotes (brief.json signals + report scope): 79 checked, 79 pass, 0 fail   [LIVE re-check against cache/pages.jsonl]

Five Forces evidence quotes, by origin:
  brief        checked  19  pass  19  fail 0   (live 19, replayed 0)
  report_page  checked  16  pass  16  fail 0   (live 16, replayed 0)
  public       checked  20  pass  20  fail 0   (live 20, replayed 0)
  match methods: {('verified', 'exact'): 55}
  quotes over 25 words: 0
  control 'changed number (public)': FAILED
  control 'reversed meaning (public)': FAILED
  control 'right quote, wrong source': FAILED
  control 'over 25 words': FAILED
  control 'changed number (report)': FAILED
  control 'reversed meaning (report)': FAILED
  control 'wrong page (report, 5 pages off)': FAILED

SUMMARY: {
 "five_forces_quotes_checked": 55,
 "five_forces_quotes_passed": 55,
 "five_forces_quotes_failed": 0,
 "by_origin": {
  "brief": {
   "checked": 19,
   "passed": 19
  },
  "report_page": {
   "checked": 16,
   "passed": 16
  }

## Stage 8: write `five_forces.json` and `five_forces.md`

Every citation carries its document title, publisher and date, the section, the PDF page (with the printed page where there is one) or URL,
the quote, and its verification result. `build_five_forces_pdf.py` reads `five_forces.json` and `reflection_five_forces.md` to build the
submission PDF.

In [12]:
REPORT_META = {s["doc_id"]: s for s in brief["sources"]}

def cite_out(c):
    if c.get("doc_id"):
        m = REPORT_META[c["doc_id"]]
        return {"source": f"{m['publisher']}, \"{m['title']}\" ({m['publication_date']})", "doc_id": c["doc_id"], "type": "paywalled report",
                "section": c["section"], "pdf_page": c["pdf_page"], "printed_page": c.get("printed_page"), "url": None,
                "quote": c["quote"], "verification": c["verification"]}
    m = src_index[c["source_id"]]
    return {"source": f"{m['publisher']}, \"{m['title']}\"", "doc_id": c["source_id"], "type": f"public ({m['kind']})",
            "section": c["section"], "pdf_page": c.get("pdf_page"), "printed_page": None, "url": m["url"], "accessed": m["accessed"],
            "quote": c["quote"], "verification": c["verification"]}

METHOD = {
    "scale": "1-5, whole numbers only. 5 = the force is strong and squeezes industry profits hard; 1 = the force is weak (written-assignment convention).",
    "polarity_conflict": ("The course reference spreadsheet (self-storage example) scores each force for attractiveness to incumbents: "
        "5 = most attractive (weak force), 1 = least attractive (strong force). The written assignment says the opposite: 5 = the force is "
        "strong and squeezes profits, 1 = weak. This tool follows the written assignment. A 4 here is a strong, profit-squeezing force; "
        "on the spreadsheet's scale the same force would be about a 2."),
    "granularity_conflict": ("The spreadsheet averages dozens of weighted sub-components into decimal force scores (its example's rivalry "
        "is 2.4167) and then averages the five forces with equal 0.2 weights into one overall score (3.39). The assignment requires whole numbers "
        "and naming the dominant forces instead of averaging. This tool uses the spreadsheet's sub-dimension categories only as a checklist to "
        "make sure each force's facts cover different points. It outputs one whole-number score per force and no sub-scores, and it never averages "
        "across forces. Where the judgment was torn between two numbers, the rationale says so."),
    "spreadsheet_used_for": "Only the standard of evidence depth: each force backed by several specific, cited facts.",
    "evidence_rules": ("Each fact has at least one citation with a verbatim quote of 25 words or fewer. A fact counts only if every quote "
        "verifies in code against cached source text; unverified facts are dropped. Each force needs at least 3 verified facts across at "
        "least 2 sub-dimensions, and its rationale must cite them by ID."),
    "who_did_what": ("Facts, claims, quotes, scores and rationales were written by Claude Code (an LLM agent) as judgment. Code fetched the "
        "public sources, mapped signals, checked coverage, verified every quote, enforced the scoring rules, and wrote these files. No code "
        "checks that a fact means what its claim says; that needs a human reader."),
}
LIMITS = [
    "Proxy industry: fractional CFO has no NAICS code. Report figures describe management consulting (54161, a superset of the chosen 541611) or accounting (54121c, the rejected neighbor), not the niche.",
    "Dated and mixed-vintage sources: IBISWorld May and August 2026, First Research July 2025 (one profitability figure is from 2023), while AI finance tools change monthly.",
    "IBISWorld 54121c contradicts itself on market size and growth ($158.4bn/1.4% vs $157.4bn/1.3%); neither figure drives a score.",
    "First Research is thin: no competitor market shares (PARTIAL/low confidence on S2), no stated industry code.",
    "No source measures buyer concentration, switching, or substitution for $2M-$20M firms specifically; the buyer and substitute scores extrapolate from industry-wide or all-business data.",
    "Vendor pages (Pilot, Intuit) are marketing claims, cited as claims, not as independent evidence of performance.",
    "Two public facts are secondary or archived: the AICPA 2025 Trends figures come via CFO Dive coverage; the BLS page was fetched from its Internet Archive snapshot because bls.gov blocks scripts.",
    "Verification proves each quote exists in its cited source; it does not prove the claim drawn from it. Scores are an LLM agent's judgment under stated rules.",
]

out_forces = []
for k, r in RESULT.items():
    fs = [f for f in FACTS if f["force"] == k]
    order = {item: i for i, (item, _) in enumerate(CHECKLIST[k])}
    fs.sort(key=lambda f: (order[f["checklist_item"]], f["fact_id"]))
    out_forces.append({
        "key": k, "name": r["name"], "score": r["score"], "rationale": r["rationale"],
        "tally_sanity_check": r["tally"], "subdimensions_covered": sorted({f["checklist_item"] for f in fs}, key=order.get),
        "facts": [{"fact_id": f["fact_id"], "label": f["checklist_item"], "subdimension_detail": f["subdimension"], "direction": f["direction"],
                   "claim": f["claim"], "origin": f["origin"], "cited_in_rationale": f["fact_id"] in r["cited_fact_ids"],
                   "citations": [cite_out(c) for c in f["citations"]]} for f in fs]})

used_sources = sorted({c.get("doc_id") or c.get("source_id") for f in FACTS for c in f["citations"]})
SOURCES = [{"id": d, "type": "paywalled report", "title": REPORT_META[d]["title"], "publisher": REPORT_META[d]["publisher"],
            "date": REPORT_META[d]["publication_date"], "how_to_find": REPORT_META[d]["how_to_find"]} if d in REPORT_META else
           {"id": d, "type": f"public ({src_index[d]['kind']})", "title": src_index[d]["title"], "publisher": src_index[d]["publisher"],
            "url": src_index[d]["url"], "accessed": src_index[d]["accessed"], "fetched_via": src_index[d].get("fetch_url")} for d in used_sources]

FF = {
    "meta": {"company": brief["meta"]["company"], "segment": "US small and mid-sized businesses, roughly $2M-$20M revenue",
             "naics": f"{brief['naics']['chosen_code']} {brief['naics']['chosen_title']} (proxy; fractional CFO has no code)",
             "generated": datetime.datetime.now().isoformat(timespec="seconds"), "input": BRIEF,
             "notebook": "five_forces.ipynb", "verification_mode": VERIFICATION_SUMMARY["report_quote_mode"]},
    "method": METHOD, "limits": LIMITS, "forces": out_forces,
    "overall_read": {"dominant_forces": [FORCES[d] for d in ov["dominant_forces"]], "text": ov["text"],
                     "scores_listed_not_averaged": {RESULT[k]["name"]: RESULT[k]["score"] for k in FORCES}},
    "least_confident": {"force": FORCES[lc["force"]], "score": RESULT[lc["force"]]["score"], "why": lc["why"],
                        "what_would_settle_it": lc["what_would_settle_it"]},
    "stage1_mapping": [{"signal_id": s, "doc_id": d, "forces": v[0], "why": v[1]} for (s, d), v in STAGE1_MAP.items()],
    "thin_before_enrichment": [FORCES[k] for k in THIN], "verification_summary": VERIFICATION_SUMMARY, "sources": SOURCES,
}
json.dump(FF, open(OUT_JSON, "w"), indent=1, ensure_ascii=False)

def md_cite(c):
    where = f"p.{c['pdf_page']}" + (f" (printed {c['printed_page']})" if c.get("printed_page") else "") if c.get("pdf_page") else ""
    loc = "; ".join(x for x in [c["section"], where, c.get("url") or ""] if x)
    return f"    - \"{c['quote']}\" ({c['source']}; {loc}) [{c['verification']}]"

L = [f"# Porter's Five Forces: {FF['meta']['company']}", "",
     f"Segment: {FF['meta']['segment']}. Industry code: {FF['meta']['naics']}. Generated {FF['meta']['generated']} by `five_forces.ipynb` from `brief.json`.", "",
     f"**Scale:** {METHOD['scale']}", "", "| Force | Score |", "|---|---|"]
L += [f"| {f['name']} | {f['score']} |" for f in out_forces]
L += ["", "## Overall read", "", f"Dominant forces: **{', '.join(FF['overall_read']['dominant_forces'])}**.", "", ov["text"], ""]
for f in out_forces:
    L += [f"## {f['name']}: {f['score']} / 5", "", f["rationale"], "",
          f"Sub-dimensions covered: {', '.join(f['subdimensions_covered'])}. Direction tally (sanity check only): "
          f"+{f['tally_sanity_check']['+']} / -{f['tally_sanity_check']['-']} / mixed {f['tally_sanity_check']['mixed']}.", ""]
    for x in f["facts"]:
        L.append(f"- **{x['fact_id']}** [{x['label']}] ({x['direction']}) {x['claim']}")
        L += [md_cite(c) for c in x["citations"]]
    L.append("")
L += ["## Least-confident force", "", f"**{FF['least_confident']['force']}** (score {FF['least_confident']['score']}). {lc['why']}", "",
      f"**What would settle it:** {lc['what_would_settle_it']}", "",
      "## Method: the two conflicts with the course spreadsheet", "", f"- **Scale polarity.** {METHOD['polarity_conflict']}",
      f"- **Granularity.** {METHOD['granularity_conflict']}", f"- **Evidence rules.** {METHOD['evidence_rules']}",
      f"- **Who did what.** {METHOD['who_did_what']}", "", "## Limits", ""] + [f"- {x}" for x in LIMITS] + [
      "", "## Verification", "",
      f"- Five Forces quotes: {VERIFICATION_SUMMARY['five_forces_quotes_checked']} checked, {VERIFICATION_SUMMARY['five_forces_quotes_passed']} passed "
      f"(report quotes: {VERIFICATION_SUMMARY['report_quote_mode']}; public quotes: live).",
      f"- First-project quotes (brief.json signals + report scope): {VERIFICATION_SUMMARY['brief_json_quotes_checked']} checked, {VERIFICATION_SUMMARY['brief_json_quotes_passed']} passed ({brief_mode}).",
      f"- Negative controls: {VERIFICATION_SUMMARY['negative_controls_rejected']} of {VERIFICATION_SUMMARY['negative_controls_run']} corrupted quotes rejected.",
      "", "## Sources", ""]
L += [f"- `{s['id']}`: {s['publisher']}, \"{s['title']}\"" + (f" ({s['date']})" if s.get("date") else f", {s['url']} (accessed {s['accessed']})") for s in SOURCES]
open(OUT_MD, "w").write("\n".join(L) + "\n")
print(f"wrote {OUT_JSON} ({os.path.getsize(OUT_JSON):,} bytes) and {OUT_MD} ({os.path.getsize(OUT_MD):,} bytes)")
print(f"{len(FACTS)} verified facts across 5 forces; {len(SOURCES)} sources; scores:",
      {f['name']: f['score'] for f in out_forces})

wrote five_forces.json (68,188 bytes) and five_forces.md (37,519 bytes)
47 verified facts across 5 forces; 10 sources; scores: {'Rivalry among existing competitors': 4, 'Bargaining power of buyers': 3, 'Bargaining power of suppliers': 4, 'Threat of new entrants': 4, 'Threat of substitutes': 3}
